In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
#Checking the number of nodes and edges of the Karate Club graph
G = nx.karate_club_graph()
nodes_ = G.number_of_nodes()
edges = G.number_of_edges()
print("The nunber of nodes: ", nodes_)
print("The number of edges: ", edges)

In [ ]:
#Visualising the graph at the first stage before performing any recursive modularity algorithms on it
pos = nx.spring_layout(G, seed = 42) #Saves the x, y coordinate in pos so that the same configuration appears everytime
plt.figure(figsize=(8,6))
nx.draw(G, pos, with_labels = True, node_color = 'pink', edgecolors = 'black', node_size = 350, font_size = 8, font_color = 'black')
plt.title("The Karate Club Graph")
plt.show()

In [ ]:
#Checking whether the edges of the Karate Club Graph have weighted edges
#We will investigate this by wrapping the G.edges(data=true) in a list and slicing some of the values from the list
l = list(G.edges(data = True))
print(l[:3])

In [ ]:
#Checking the order of nodes 
w = list(G.nodes())
print(w)

In [ ]:
#Since it is in order, while building the adjacency matrix A we can directly assign nodelist = G.nodes as the ith node will correspond to the ith row
A = nx.to_numpy_array(G, nodelist = G.nodes(), dtype = float, order = None, weight = None, nonedge = 0.0)
#Checking the matrix A 
print(A.shape) #checks how many rows and columns are there, should return 34 x 34
print(np.unique(A)) #checks for unique values in the matrix, should only return 0 and 1
print(np.allclose(A, A.T)) #checks if A and A transpose are equal, should be equal as A is an undirected graph, hence A = A^T
print(A.sum()) #finds the sum along rows and columns and since each edge contributes degree to two nodes, the total number of edges are 2m 

<div style="font-family: 'EB Garamond', Garamond, serif; font-size: 14px;">

# Method

## Disclaimer

In my method I have used un-weighted graphs. 

## The basic idea

I've made an attempt to explain how we split the Karate Club community into sub-communities, hopefully in a straightforward and easy manner. 

The methodology is based on something called Modularity. It basically lets you decide where to put a node into depending on its actual connections as compared to how many connections it could've had in a random network which preserved its degree. 

## The matrix

The first step in applying the above is to build the modularity matrix B (Newman, 2006).
$$B_{ij} = A_{ij} - \frac{k_i k_j}{2m}, \qquad \text{i.e.} \qquad B = A - \frac{k\,k^{\top}}{2m}$$

where $A$ is the adjacency matrix, $k_i = \sum_j A_{ij}$ is the degree of node $i$, and $m$ is the number of edges.

Once we have the modularity matrix B, the next step is to maximise the modularity score, Q.
$$Q = \frac{1}{4m}\, s^{\top} B\, s$$

To maximise this we must select our S carefully. The ideal way to do it would be to assign +1 or -1 to each node (+1 would mean it belongs to a certain community and -1 would mean it belongs to anotehr certain community) so that the modularity score is maximised for the split. However, to compute such a thing would be something next to impossible as it would require us to check $2^n$ possibilites. In this case n would equal 34 and that would mean computing $2^{34}$ possibilities. Since thats not possible. We do something different. We calculate the leading eigenvector of B, say u and we assign s a value of +1 for every $u_i$ > 0 and -1 for every $u_i$ otherwise. Our s vector would look something like: 
$$
s_i =
\begin{cases}
+1 & \text{if } (u_1)_i > 0,\\
-1 & \text{otherwise.}
\end{cases}
$$

# Recursion 

We split the communities formed using the method above, by building a B for every community that is formed. We stop when the leading eigenvalue is non-positive (Newman, 2006). Additionally, a split with an empty side is treated as an indivisible community.
The new B, Q and the stopping and the stopping condition for a community C is as follows: 

For a community $C$, use the block $B^{(C)}$ of $B$ (rows and columns in $C$), with leading eigenvalue $\lambda_1^{(C)}$. The change in modularity from splitting $C$ is

$$\Delta Q = \frac{1}{4m}\, s^{\top} B^{(C)} s$$

and the stopping rule is: split $C$ only if $\lambda_1^{(C)} > 0$.
</div>

In [ ]:
#Now that we have A, we can go to the next step and build B. Building B involves first building k, the degree vector
k = np.zeros(nodes_)
for i in range(34):
    k[i] = np.sum(A[i])
print(k[6], k[0], k[33]) #checking if the vector k is made correctly
print(k.shape) #checking the dimensions of the vector k

In [ ]:
#now we define the 'm' to be the number of edges of the graph
m = edges # edges have been defined earlier on in teh notebook
#2m is the sum of all the degrees of the nodes
#now we build the baseline
baseline = (np.outer(k,k))/(2*m) #using np.outer gives us a matrix of k[i]*k[j] both i,j belonging to matrix A
B = A - baseline
#now we will run some checks on B
print(B.shape)
print(np.allclose(B, B.T))
#To check B1 = 0 we need to build a ones vector of the size 34x1
ones = np.ones(34)
print(B @ ones)
print(B.sum())

In [ ]:
#first we store the eigenvalues and eigenvectors of B in variables to perform the eigenvalue test
u, v = np.linalg.eigh(B)
u1 = u[-1]
v1 = v[:,-1]
print(u1) #sanity check 

In [ ]:
#building the assignment vector S. S is defined to +1 for (v1)i>0 and -1 otherwise
#initialise S
s = np.zeros(34)
for i in range(34):
    if v1[i] > 0 :
        s[i] = 1
    else:
        s[i] = -1
print(s)
#some checks for s
print(s.shape)
print((s==1).sum(), (s==-1).sum())
print(np.unique(s))

In [ ]:
#now we calculate the modularity score Q, if Q > 0 then we know the split is meaningful, otehrwise the split renders useless. 
q = (1/(4*m))*(s.T)@(B)@(s)
print(q)

In [ ]:
community = list(G.nodes())
print(community)

In [ ]:
#the first split, building the code for the first split 
community = list (G.nodes())
B_= np.zeros((len(community), len(community)))
for i in range(len(community)):
    for j in range(len(community)):
        p = community[i]
        t = community[j]
        B_[i][j]= A[p][t] - (k[p]*k[t])/(2*m)
def eigen_test(B_):
    global u_1, v_1
    u_, v_ = np.linalg.eigh(B_) 
    u_1 = u_[-1]
    v_1 = v_[:,-1]
    if u_1 > 0:
        return True
    else : 
        return False
def split(community):
    if (eigen_test(B_) == True):
        s_ = np.zeros(len(community))
        comm1 = []
        comm2 = []
        for i in range(len(community)):
            if v_1[i] > 0 :
                s_[i] = 1
            else:
                s_[i] = -1
            if s_[i] == 1:
                comm1.append(community[i])
            else :
                comm2.append(community[i])
    return comm1, comm2
print(split(community))

In [ ]:
def eigen_test(B_c): #code block defining eigen_test
    global u_1, v_1 #global because if used anywhere else in the code, it must be accessible 

    u_, v_ = np.linalg.eigh(B_c)
    u_1 = u_[-1] #corresponds to largest eigen value 
    v_1 = v_[:, -1] #corresponds to eigenvector that corresponds to largest eigenvalue 

    if u_1 > 1e-10:
        return True
    else:
        return False


def split(current_comm): #code block defining split
    if eigen_test(B_c):
        s_ = np.zeros(len(current_comm)) #constructing the assignment vector based off eigenvector 
        comm1 = []
        comm2 = []

        for i in range(len(current_comm)):
            if v_1[i] > 0:
                s_[i] = 1
            else:
                s_[i] = -1

            if s_[i] == 1:
                comm1.append(current_comm[i]) #just a way to split the current_comm into two sub-communities comm1 and comm2
            else:
                comm2.append(current_comm[i])

        return comm1, comm2
    else:
        return None


waiting_list = [list(G.nodes())] #updates to list of split subcommunities
final_list = [] #constains a list of the communities that cannot be split further 
history_of_snapshots = [[list(G.nodes)]] #has elements that correspond to all the splits that happened every iteration, including iteration 0, which is 1 community

while len(waiting_list) > 0: #this basically means, the code below will run until the length of waiting_list is nonzero, which means until it contains communities left to be tested first and then divided 
    current_comm = waiting_list.pop(0) #waiting_list is a list of communities, we remove the first community and assign it to current_comm, on which we run test and then divide

    B_c = np.zeros((len(current_comm), len(current_comm))) #creates B_c for every iteration with updated current length
    for i in range(len(current_comm)):
        for j in range(len(current_comm)):
            p = current_comm[i]
            t = current_comm[j]
            B_c[i][j] = A[p][t] - (k[p] * k[t]) / (2 * m)

    if eigen_test(B_c): #the code block below if will run only eigen_test(B_c) returns True
        comm1, comm2 = split(current_comm)

        if comm1 and comm2: #this statement is only true if both comm1 and comm2 are nonempty 
            waiting_list.append(comm1)
            waiting_list.append(comm2)
            history_of_snapshots.append(final_list + waiting_list)
        else:
            final_list.append(current_comm) #this basically means if current is non-divisible, jst add it to final_list
    else: #if eigen_test(B_) returns false, it will jump to this block
        print(current_comm, "cannot be split further") 
        final_list.append(current_comm)
print(final_list)

In [ ]:
#this code block is to print the Q values for each iteration using the nx.community.modularity function, I used my logic and took some help from Claude to write the code block below
q_values = []
for i, snapshot in enumerate(history_of_snapshots):
    partition = [set(c) for c in snapshot]
    q = nx.community.modularity(G, partition, weight = None)
    q_values.append(q)
    print(f"Iteration {i} : {len(snapshot)} communities, Q = {q:.4f}")   

In [ ]:
#now time to visualise each split, and since i have a list that contains every split called history_of_snapshots, we'll iterate over that list and visualise each split
#first we assign a community label to each of the nodes in G 
nodes = list(G.nodes())

communities_to_check = final_list.copy() #creates a copy of the final_list so that the changes i make on it, dont change final_list
community1 = communities_to_check.pop(0) #extracts all the communities 
community2 = communities_to_check.pop(0)
community3 = communities_to_check.pop(0)
community4 = communities_to_check.pop(0)
community5 = communities_to_check.pop(0)

community_labels = {}

for i in range(len(nodes)):
    node = nodes[i]

    if node in community1:
        community_labels[node] = 0
    elif node in community2:
        community_labels[node] = 1
    elif node in community3:
        community_labels[node] = 2
    elif node in community4:
        community_labels[node] = 3
    elif node in community5:
        community_labels[node] = 4

node_colors = [community_labels[node] for node in nodes]
#the above code was thought of first, and when I ran it i realised the code only visualised the final list of communities. To be able to visualise every split, i then thought of using a while loop, and with some assitance from AI i was able to impolement the code below.

In [ ]:
plt.figure(figsize=(8,6))
nx.draw(G, pos, with_labels = True, node_color = node_colors, cmap = "tab10", node_size = 350, font_size = 8, font_color = 'black')
plt.title("The final number of communities in the Karate Club Graph")
plt.show()

In [ ]:
#below is the code in which i used some AI to help me with syntax and the draw function. The logic was mine.
history_of_splits = history_of_snapshots.copy()

step = 0

while len(history_of_splits) > 0:
    communities_to_check = history_of_splits.pop(0)

    # Give each node the number of the community it belongs to
    community_labels = {}
    next_label = len(final_list)
    for community in communities_to_check:
        if community in final_list:
            label = final_list.index(community)
        else:
            label = next_label
            next_label = next_label +1
        for node in community:
            community_labels[node] = label
    # Put the labels in the same order as the nodes in G
    node_colors = [community_labels[node] for node in nodes]

    plt.figure(figsize=(10, 8))

    nx.draw(
        G,
        pos,
        with_labels = True,
        nodelist=nodes,
        node_color=node_colors,
        cmap="tab10",
        vmin=0,
        vmax=9,
        node_size=350
    )
    plt.title(f"Iteration {step}: {len(communities_to_check)} communities")
    plt.show()
    step = step+1


<div style="font-family: 'EB Garamond', Garamond, serif; font-size: 14px;">

# Results 

After the 4th iteration I ended up with 5 communities. 
The modularity score Q changed as follows with each iteration:

- Iteration 0 : 1 communities, Q = 0.0000
- Iteration 1 : 2 communities, Q = 0.3715
- Iteration 2 : 3 communities, Q = 0.3898
- Iteration 3 : 4 communities, Q = 0.4081
- Iteration 4 : 5 communities, Q = 0.4094
</div>

In [ ]:
#the below code block was created with assistance from Claude
def print_metrics(iteration, deg, bet, clo, clu):
            print(f"\n===== Iteration {iteration} =====")
            print(f"{'Node':>4} {'Degree':>8} {'Between.':>9} {'Closeness':>10} {'Cluster.':>9}")
            for node in sorted(deg):
                print(f"{node:>4} {deg[node]:>8.3f} {bet[node]:>9.3f} {clo[node]:>10.3f} {clu[node]:>9.3f}")
                
metrics = {"degree":{}, "betweenness":{}, "closeness":{}, "clustering":{}}

for i, communities in enumerate(history_of_snapshots):
    deg = {}
    bet = {}
    clo = {}
    clu = {}
    for community in communities:
        community_graph = G.subgraph(community)
        deg.update(nx.degree_centrality(community_graph))
        bet.update(nx.betweenness_centrality(community_graph))
        clo.update(nx.closeness_centrality(community_graph))
        clu.update(nx.clustering(community_graph))
    metrics["degree"][i] = deg
    metrics["betweenness"][i] = bet
    metrics["closeness"][i] = clo
    metrics["clustering"][i] = clu
    print_metrics(i, deg, bet, clo, clu)

In [ ]:
# the list comprehension that builds the array in this cell was written with assistance from Claude
#this whole code block was to get make a variable that stores, the node values for degree centrality, per iteration so that we can use that to make our evolution plots
iters = sorted(metrics["degree"].keys())
list_of_node_values_per_iteration = []
for iteration_keys in iters:
    list_of_node_values_per_iteration.append([metrics["degree"][iteration_keys][node] for node in sorted(metrics["degree"][iteration_keys])])
array_of_node_values = np.array(list_of_node_values_per_iteration)
print(array_of_node_values)
print(array_of_node_values.shape)
#sanity checks 
print(array_of_node_values[4][0])

In [ ]:
#now comes the plotting of the graph, this doesn't show which line represents which node. To show that I used AI in for help in the next cell
plt.figure(figsize=(8,6))
plt.plot(iters,array_of_node_values)
plt.xlabel("Iteration")
plt.ylabel("Degree centrality")
plt.title("Degree centrality evolution")
plt.show()

In [ ]:
#used ai to make this code as i wasnt able to understand how to add a legend to the diagram, and how to have 34 different colors. I could've tried but due to lack of time i went ahead with ai for this code
plt.figure(figsize=(20, 18))
colors = plt.cm.nipy_spectral(np.linspace(0, 0.95, 34))   # 34 different colours

for node in range(34):
    plt.plot(iters, array_of_node_values[:, node], color=colors[node], label=str(node))
    for it, value in zip(iters, array_of_node_values[:, node]):   # node number written at each point instead of a dot
        plt.text(it, value, str(node), fontsize=10, color=colors[node], ha="center", va="center")

plt.xlabel("Iteration", fontsize = 20)
plt.ylabel("Degree centrality", fontsize = 20)
plt.title("Degree centrality evolution", fontsize = 24)
plt.xticks(iters)
plt.legend(title="Node", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=16, title_fontsize = 20, labelspacing = 0.4, handlelength = 1.2, columnspacing = 0.8,)
plt.tight_layout()
plt.show()

In [ ]:
# the list comprehension that builds the array in this cell was written with assistance from Claude
#this code is now for betweenness centrality
iters_bet = sorted(metrics["betweenness"].keys())
list_of_node_vals_per_iter = []
for iter_keys in iters_bet:
    list_of_node_vals_per_iter.append([metrics["betweenness"][iter_keys][n] for n in sorted(metrics["betweenness"][iter_keys])])
array_of_node_vals = np.array(list_of_node_vals_per_iter)
print(array_of_node_vals)
print(array_of_node_vals.shape)
#sanity checks 
print(array_of_node_vals[4][0])

In [ ]:
# plotting code (34-colour legend and node numbers written on the lines) was written with assistance from Claude
#plotting betweenness centrality 
plt.figure(figsize=(20, 18))
colors = plt.cm.nipy_spectral(np.linspace(0, 0.95, 34))   # 34 different colours

for n in range(34):
    plt.plot(iters, array_of_node_vals[:, n], color=colors[n], label=str(n))
    for it, value in zip(iters, array_of_node_vals[:, n]):   # node number written at each point instead of a dot
        plt.text(it, value, str(n), fontsize=10, color=colors[n], ha="center", va="center")

plt.xlabel("Iteration", fontsize = 20)
plt.ylabel("Betweenness centrality", fontsize = 20)
plt.title("Betweenness centrality evolution", fontsize = 24)
plt.xticks(iters_bet)
plt.legend(title="Node", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=16, title_fontsize = 20, labelspacing = 0.4, handlelength = 1.2, columnspacing = 0.8,)
plt.tight_layout()
plt.show()

In [ ]:
# the list comprehension that builds the array in this cell was written with assistance from Claude
#this code is now for closeness centrality 
iters_clo = sorted(metrics["closeness"].keys())
list_of_node_vals_per_iter_clo = []
for ite_keys in iters_clo:
    list_of_node_vals_per_iter_clo.append([metrics["closeness"][ite_keys][n_] for n_ in sorted(metrics["closeness"][ite_keys])])
array_node_vals = np.array(list_of_node_vals_per_iter_clo)
print(array_node_vals)
print(array_node_vals.shape)
#sanity checks 
print(array_node_vals[4][0])

In [ ]:
# plotting code (34-colour legend and node numbers written on the lines) was written with assistance from Claude
#plotting closeness centrality 
plt.figure(figsize=(20, 18))
colors = plt.cm.nipy_spectral(np.linspace(0, 0.95, 34))   # 34 different colours

for n_ in range(34):
    plt.plot(iters_clo, array_node_vals[:, n_], color=colors[n_], label=str(n_))
    for it, value in zip(iters_clo, array_node_vals[:, n_]):   # node number written at each point instead of a dot
        plt.text(it, value, str(n_), fontsize=10, color=colors[n_], ha="center", va="center")

plt.xlabel("Iteration", fontsize = 20)
plt.ylabel("Closeness centrality", fontsize = 20)
plt.title("Closeness centrality evolution", fontsize = 24)
plt.xticks(iters_clo)
plt.legend(title="Node", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=16, title_fontsize = 20, labelspacing = 0.4, handlelength = 1.2, columnspacing = 0.8,)
plt.tight_layout()
plt.show()

In [ ]:
# the list comprehension that builds the array in this cell was written with assistance from Claude
#this code is now for clustering 
iters_clu = sorted(metrics["clustering"].keys())
node_values_per_iteration = []
for i_keys in iters_clu:
    node_values_per_iteration.append([metrics["clustering"][i_keys][node_] for node_ in sorted(metrics["clustering"][i_keys])])
array_of_values = np.array(node_values_per_iteration)
print(array_of_values)
print(array_of_values.shape)
#sanity checks 
print(array_of_values[4][0])

In [ ]:
# plotting code (34-colour legend and node numbers written on the lines) was written with assistance from Claude
#plotting clustering 
plt.figure(figsize=(20, 18))
colors = plt.cm.nipy_spectral(np.linspace(0, 0.95, 34))   # 34 different colours

for node_ in range(34):
    plt.plot(iters_clu, array_of_values[:, node_], color=colors[node_], label=str(node_))
    for it, value in zip(iters_clu, array_of_values[:, node_]):   # node number written at each point instead of a dot
        plt.text(it, value, str(node_), fontsize=10, color=colors[node_], ha="center", va="center")

plt.xlabel("Iteration", fontsize = 20)
plt.ylabel("Clustering coefficient", fontsize = 20)
plt.title("Clustering coefficeint evolution", fontsize = 24)
plt.xticks(iters_clu)
plt.legend(title="Node", bbox_to_anchor=(1.02, 1), loc="upper left", ncol=2, fontsize=16, title_fontsize = 20, labelspacing = 0.4, handlelength = 1.2, columnspacing = 0.8,)
plt.tight_layout()
plt.show()

<div style="font-family: 'EB Garamond', Garamond, serif; font-size: 14px;">

# Short Discussion


## Disclosure

All these metrics are computed on subgraphs of the size of the communities as they were formed. These metrics were computed on nodes belonging to different depths of splitting. However, these metrics were calculated for every split that happened and they capture how these metrics evolved for each node as the graph split into 5 communities.

## Degree centrality

Nodes 0, 11 and 33 have the highest Degree centrality after the 4 iterations. Node 11, started off with the lowest degree centrality, but ended up with a high degree centrality score. Node 11 jumps to a degree centrality of 1, at the 4th iteration, when it becomes a singleton community. The NetworkX library handles this case by assigning 1 to its degree centrality score. [Because for that community, Degree centrality becomes = k\_i/(n-1), where k\_i = 0, and n-1 = 1-1=0, therefore 0/0, and hence it assigns a value of 1]. Meanwhile nodes 0 and 33 have a high degree of centrality as they have high connectivity to other nodes.

Most of the nodes continue to raise their degree of centrality as the number of iterations increases. This makes intuitive sense as it would mean that the split is beneficial and increases the relative connectivity of each node, because as the iterations happen, the community size reduces.

## Betweenness Centrality

After the first split nodes 0 and 33 had high betweenness centrality as they were nodes that connected many other nodes through themselves. As nodes 31 and 24 became central to small communities, their betweenness centrality score increased, after iteration 2. This indicates that they become better bridges that connect edge nodes , and also means they became central to their groups.

Meanwhile, the betweenness centrality dipped noticeably for node 0 and slightly for node 33, which indicated that they no longer remained as bridges for certain edge nodes they previously connected.

## Closeness Centrality

Almost all nodes increased their closeness centrality scores, except node 11 that ends lower. A few nodes dip in intermediate steps.

Since node 11 is only connected to node 0, the only way it could reach other nodes in the network was through node 0, hence it makes sense that as soon as it got detached from node 0, its closeness centrality reduced. Nodes 0, 33, 1 and 32 have high closeness centrality after the 4 iterations, indicating that they are good “spreaders” in the network.

One might wonder why the closeness centrality fell immediately for node 11 after the second split. That's because the nodes in its community were not connected to it. Hence its closeness scores 0 from the second split onwards.

## Clustering coefficient

At the end of the four iterations nodes 0 and 33 have low clustering coefficients, because they are hubs. Another effect of being a hub is having high betweenness centrality. These two happen to any node that acts as hubs. However, nodes 7,8, 12 to 23 (all of them) , 25,26, and 30 have clustering coefficients of 1. Which means all of their neighbours are connected to each other. This indicates that these nodes are embedded in tightly knit communities.

## General trends

Throughout the iterations, nodes 0, 32 and 33 maintain high scores of degree, betweenness and closeness centralities. While node 1 joins them after iteration 2.

Nodes 0 and 33 have high scores for closeness and degree centrality, while they have low clustering coefficients. Node 33 keeps the highest betweenness score at the end of 4 iterations, whereas the betweenness of node 0 falls from 0.690 to 0.354 and is overtaken by nodes 24 and 31. Nodes 32 and 31 have similar closeness scores at the end of 4 iterations (0.857 and 0.800), but node 31 pulls ahead in betweenness (0.500 against 0.258) once it becomes central to its small community. Node 1 has low betweenness at the start (0.054), rises to 0.215 by iteration 2, and scores well in degree and closeness metrics.

## How community structure affects metrics

From the trends it is quite clear that as community sizes reduce, for most nodes, the scores for degree and closeness increase as their relative metrics change . For example, by reducing the community size for node 0, the number of connections it had with nodes versus the number of connections it could have become closer to each other, hence increasing its score for degree centrality.

However there is no trend as such for betweenness centrality. This increases if nodes become central to communities formed and scores low for nodes that stop being bridges for certain edge nodes. One can clearly see this is true by examining the “Betweenness centrality section” in this discussion.

## References

Newman, M. E. J. (2006). Modularity and community structure in networks. *Proceedings of the National Academy of Sciences*, *103*(23), 8577–8582. [https://doi.org/10.1073/pnas.0601602103](https://doi.org/10.1073/pnas.0601602103)

## AI disclosure

This notebook was created with assistance from Claude and ChatGPT. I used these to help me with the plotting of figures, checking for errors in code, learning the new functions of NetworkX library, learning how to implement Numpy functions better and sometimes helping sort out my logic.

Additionally, Claude also wrote some code and I have mentioned them in the comments. 

Claude also checked the factual accuracy of my discussion section. I made Claude use the tables that were printed for nodes to verify my claims in the discussion section. It also helped me format the markdown tab. 
</div>